# Phase 4 Pipeline Walkthrough
**Facial Color without a ColorChecker — step by step**

Emily Rooney · June 2026

---

This notebook walks through the **Phase 4 chart-free flash/no-flash skin color pipeline**
exactly as described in the write-up (*Facial Color without Checker*, June 2026).
Each cell corresponds to one numbered step.

**Big picture in one sentence:**
Take a pair of iPhone photos (one with flash, one without),
cancel out the room lighting mathematically,
extract the skin color from the cheek,
and compare it to what a contact spectrophotometer (FitSkin) measured on the same spot.

---

## Pipeline overview
```
── OFFLINE (done once, saved as files) ────────────────────────────
  Step 1.  Measure iPhone spectral sensitivity  → q_k(λ)
  Step 2.  Measure iPhone flash spectrum        → CCT 4923 K
  Step 3.  Photograph ColorChecker, fit matrix  → M  (camera RGB → XYZ)
  Step 4.  Derive per-participant exposure anchor → s_exp

── INFERENCE (runs per flash/no-flash pair) ───────────────────────
  Step 5.  Demosaic both DNG files              → linear camera RGB  A, B
  Step 6.  ECC register flash → no-flash        → aligned B
  Step 7.  Match flash exposure over skin       → scaled B
  Step 8.  Reflectance proxy  R = √(A ⊙ B)     → apply s_exp
  Step 9.  R  ──M──→  XYZ  ──→  CIELAB         → Lab per pixel
  Step 10. Trimmed mean over cheek ROI          → (L*, a*, b*) prediction
  Step 11. CIEDE2000 vs FitSkin                 → ΔE₀₀
```

## 0 — Install dependencies
Run this cell once when you first open the notebook in Google Colab.

In [ ]:
# rawpy      — reads iPhone DNG (raw sensor) files
# opencv     — ECC image alignment
# mediapipe  — face mesh for locating the cheek region
# numpy      — all numerical operations
# matplotlib — plots
!pip install rawpy opencv-python-headless mediapipe numpy matplotlib --quiet

In [ ]:
import json
import numpy as np
import cv2
import rawpy
import mediapipe as mp
import matplotlib.pyplot as plt
from pathlib import Path

# ── calibration file locations ───────────────────────────────────────────────
# git clone puts everything inside a 'Fitskin/' subfolder,
# so all paths start with 'Fitskin/'
REPO_DIR = Path('Fitskin')
CAL_DIR  = REPO_DIR / 'calibration' / 'tier3_affine'
CAL_JSON = CAL_DIR / 'iphone_calibration_bundle.json'

# Load the bundle once — it contains all offline calibration results
with open(CAL_JSON) as f:
    cal = json.load(f)

print('Calibration bundle loaded.')
print('Keys available:', list(cal.keys())[:10], '...')

---
## Part 1 — Offline Calibration

These four steps were carried out **once in the lab** and saved into the calibration bundle.
During evaluation we simply load the results — no ColorChecker needs to be in the photo.

### Step 1 — Spectral sensitivity of the iPhone camera

A **Newport Cornerstone 260 monochromator** illuminated the iPhone 17 Pro sensor
with narrow-band light from 400 to 700 nm (10 nm steps, 93 wavelengths total).
Three repeat DNG captures were taken at each wavelength.
A PR-250 spectroradiometer measured the radiance $L(\lambda_n)$ simultaneously.

The per-channel sensitivity was computed as:
$$q_k(\lambda_n) = \frac{\bar{x}_k(\lambda_n)}{L(\lambda_n)}$$

where $\bar{x}_k$ is the mean pixel value in channel $k$ (R, G, or B) over a
central region of interest. Each channel is then normalised to a maximum of 1.

**Why we need this:** The sensitivity curves tell us how the camera 'sees' different
wavelengths of light — essential for converting camera values into true colour coordinates.

In [ ]:
# Load spectral sensitivity from the calibration bundle
# Shape: (N_wavelengths, 3)  — one column per colour channel (R, G, B)
spectral_sensitivity = np.array(cal['spectral_sensitivity_rgb'])
wavelengths_nm       = np.array(cal['wavelengths_nm'])           # e.g. [400, 410, ..., 700]

print(f'Wavelength range:      {wavelengths_nm[0]:.0f} – {wavelengths_nm[-1]:.0f} nm')
print(f'Number of steps:       {len(wavelengths_nm)}')
print(f'Sensitivity array shape: {spectral_sensitivity.shape}  (wavelengths × channels)')

# ── Plot ──────────────────────────────────────────────────────────────────────
plt.figure(figsize=(8, 3))
for ch, color, label in zip(range(3), ['red', 'green', 'blue'], ['Red', 'Green', 'Blue']):
    plt.plot(wavelengths_nm, spectral_sensitivity[:, ch], color=color, label=f'{label} channel')
plt.xlabel('Wavelength (nm)')
plt.ylabel('Sensitivity (normalised to 1)')
plt.title('iPhone 17 Pro spectral sensitivity')
plt.legend()
plt.tight_layout()
plt.show()

print('\nPeak sensitivity wavelengths:')
for ch, name in enumerate(['Red', 'Green', 'Blue']):
    peak_wl = wavelengths_nm[np.argmax(spectral_sensitivity[:, ch])]
    print(f'  {name}: {peak_wl:.0f} nm')

### Step 2 — iPhone flash spectral power distribution (SPD)

The iPhone 17 Pro video flash was measured with a **UPRtek MK350 spectroradiometer**
in a darkened room (all ambient lights off, only flash running).
Six measurements were averaged.

Result: the flash has a correlated colour temperature (CCT) of **4923 K**.
This is slightly warm — between D65 daylight (6500 K) and tungsten (2700 K).
It differs from the 5500 K Planckian assumption used in earlier pipeline phases.

**Why this matters:** The flash spectrum determines how the skin reflectance translates
into camera values in the flash frame. Knowing the CCT lets us relate camera readings
to device-independent colour coordinates.

In [ ]:
# Flash characterisation is stored in the calibration bundle
flash_cct_k     = cal['flash_cct_k']         # correlated colour temperature in Kelvin
flash_rgb_lin   = cal['flash_rgb_linear']     # how the flash appears in linear camera RGB
flash_xyz       = cal['flash_xyz']            # flash in CIE XYZ (device-independent)

print(f'Flash CCT:  {flash_cct_k:.0f} K  (Phase 3 assumed 5500 K)')
print(f'Flash in camera RGB: R={flash_rgb_lin[0]:.4f}  G={flash_rgb_lin[1]:.4f}  B={flash_rgb_lin[2]:.4f}')
print(f'Flash in XYZ:        X={flash_xyz[0]:.4f}  Y={flash_xyz[1]:.4f}  Z={flash_xyz[2]:.4f}')
print()
print('Colour balance of flash (relative to green):')
print(f'  R/G = {flash_rgb_lin[0]/flash_rgb_lin[1]:.3f}  (>1 → warm / reddish)')
print(f'  B/G = {flash_rgb_lin[2]/flash_rgb_lin[1]:.3f}  (<1 → less blue than D65)')

# ── Plot the flash spectral power distribution ──────────────────────────────
if 'flash_spd_wl_nm' in cal and 'flash_spd_power' in cal:
    spd_wl  = np.array(cal['flash_spd_wl_nm'])
    spd_pow = np.array(cal['flash_spd_power'])
    plt.figure(figsize=(8, 3))
    plt.plot(spd_wl, spd_pow / spd_pow.max(), 'darkorange')
    plt.xlabel('Wavelength (nm)')
    plt.ylabel('Relative power')
    plt.title(f'iPhone flash SPD  (CCT = {flash_cct_k:.0f} K)')
    plt.tight_layout()
    plt.show()

### Step 3 — ColorChecker → Affine colour correction matrix M

Six flash/no-flash DNG pairs were selected where a **ColorChecker Classic 24**
was in the no-flash frame. The OpenCV MCC detector found the chart automatically.
Linear camera RGB values were sampled from each of the 24 patches.

We fitted an **affine 4×3 matrix M** by ordinary least squares:
$$\begin{bmatrix} X \\ Y \\ Z \end{bmatrix} = M \begin{bmatrix} R \\ G \\ B \\ 1 \end{bmatrix}$$

The fourth column (the `1`) adds an **offset term** — this lets M correct for
black-level and sensor bias that a simple 3×3 matrix cannot account for.

**Why not a weighted or Huber fit?** We tested both — they increased error on held-out
patches, so ordinary least squares was used as the production fit.

Once we have M, we can convert any camera RGB value to XYZ — a device-independent
colour space that can be compared directly to FitSkin scanner output.

In [ ]:
# Load the affine colour correction matrix M
# Shape: (4, 3) — stored as [R; G; B; 1] on the LEFT, XYZ on the RIGHT
# Usage: xyz = [R, G, B, 1] @ M   (a row vector times M gives XYZ)
# Row 3 (last row) is the additive offset term
M = np.array(cal['camera_rgb_to_xyz_affine'])  # shape (4, 3)

print(f'M shape: {M.shape}  → usage: [R, G, B, 1] @ M = [X, Y, Z]')
print()
print('Affine CCM  M:')
print('              X          Y          Z')
for row_name, row in zip(['R', 'G', 'B', 'offset'], M):
    vals = '   '.join(f'{v:+.5f}' for v in row)
    print(f'  {row_name:6s}  {vals}')

print()

# Sanity check: what does a bright grey pixel (R=G=B=0.9) map to?
# Append 1 for the affine offset term, then multiply on the LEFT
test_rgb = np.array([0.9, 0.9, 0.9, 1.0])   # shape (4,)
test_xyz = test_rgb @ M                       # (4,) @ (4,3) → (3,)
print('Sanity check — grey pixel RGB=[0.9, 0.9, 0.9] maps to:')
print(f'  XYZ = [{test_xyz[0]:.4f}, {test_xyz[1]:.4f}, {test_xyz[2]:.4f}]')
print('  (Expected roughly proportional to D65 white [0.95, 1.00, 1.09] for a neutral grey)')

### Step 4 — Per-participant exposure anchor s_exp

Even with the colour matrix M, the camera's **absolute brightness** differs between
participants because they were photographed in different rooms with different ambient light.

For each participant's ColorChecker training sessions, we found the scalar **s_exp**
that maps the white patch's camera value to canonical D65 white (Y = 1.0):

$$s_{\text{exp}} = \frac{Y_{\text{D65, white}}}{Y_{\text{camera white patch}}}$$

| Participant | s_exp | Meaning |
|---|---|---|
| P1 (Emily) | 0.961 | Camera slightly overexposed — scale down |
| P2 (Liki)  | 1.393 | Camera underexposed — needs brightness boost |

This anchor is applied at inference **without** needing a chart in the evaluation photos.

In [ ]:
# Load exposure anchors from the calibration bundle
# Keys are participant IDs: 'P1', 'P2', ...
exposure_anchors = cal['exposure_anchor_by_participant']

print('Per-participant exposure anchors:')
for pid, s_exp in exposure_anchors.items():
    meaning = 'overexposed (scale down)' if s_exp < 1 else 'underexposed (scale up)'
    print(f'  {pid}:  s_exp = {s_exp:.4f}   →  {meaning}')

print()
print('These scalars are derived from ColorChecker sessions.')
print('They are applied during inference without any in-frame chart.')

---
## Part 2 — Chart-Free Inference

These steps run on **every evaluation flash/no-flash pair**.
No ColorChecker is needed in the frame.

**Set your file paths here:**

In [ ]:
# ── EDIT THESE FOR YOUR DATA ──────────────────────────────────────────────────
# Upload your DNG files using the Colab Files panel (folder icon on the left),
# then update the filenames below.
PATH_NOFLASH  = Path('/content/Emily_Trial_1_NoFlash.DNG')   # no-flash DNG
PATH_FLASH    = Path('/content/Emily_Trial_1_Flash.DNG')     # flash DNG (same trial)
PARTICIPANT   = 'P1'                                         # 'P1' or 'P2'

# FitSkin scanner reading for this trial (from your scan export CSV)
FITSKIN_L = 62.4   # L* from FitSkin cheek scan
FITSKIN_A =  8.1   # a*
FITSKIN_B = 18.3   # b*
# ─────────────────────────────────────────────────────────────────────────────

s_exp = exposure_anchors[PARTICIPANT]   # look up anchor for this participant
print(f'Participant:     {PARTICIPANT}')
print(f'Exposure anchor: s_exp = {s_exp:.4f}')

### Step 5 — RAW DNG demosaicing

The iPhone saves photos as **DNG (Digital Negative)** files — raw sensor data,
not the processed JPEG you see on the phone screen.
The sensor records only **one colour channel per pixel** in a Bayer grid pattern
(RGGB). `rawpy` reconstructs a full 3-channel RGB image from that pattern.

Key settings:
- `user_wb=[1,1,1,1]` — unity white balance: **no** automatic white balance
  correction. We want the true sensor values, not a pre-adjusted version.
- `gamma=(1,1)` — linear output (no gamma curve applied).
- Normalise each frame to its 99.5th percentile so values fall in [0, 1],
  avoiding hot pixels from dominating the scale.

In [ ]:
def load_dng_linear(path, half_size=True):
    """Read a DNG file. Returns a linear RGB image with values in [0, ~1]."""
    with rawpy.imread(str(path)) as raw:
        rgb = raw.postprocess(
            use_camera_wb=False,    # disable iPhone auto white balance
            use_auto_wb=False,
            user_wb=[1, 1, 1, 1],   # unity gains: no per-channel adjustment
            gamma=(1, 1),           # no gamma — output stays linear
            no_auto_bright=True,    # do not adjust brightness
            output_bps=16,          # 16-bit integers from demosaic
            half_size=half_size,    # halve resolution for faster processing
        ).astype(np.float64)

    # Normalise: divide by the 99.5th percentile value
    # This maps nearly all pixels to [0, 1] while ignoring rare hot pixels
    scale = np.percentile(rgb, 99.5)
    rgb = rgb / scale
    return rgb


print('Loading no-flash frame (A)...')
A = load_dng_linear(PATH_NOFLASH)   # ambient light only
print(f'  Shape: {A.shape}   min={A.min():.4f}   max={A.max():.4f}')

print('Loading flash frame (B)...')
B = load_dng_linear(PATH_FLASH)     # ambient + flash
print(f'  Shape: {B.shape}   min={B.min():.4f}   max={B.max():.4f}')

# Show the two frames for visual inspection
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, img, title in zip(axes,
                           [A, B],
                           ['No-flash frame  A', 'Flash frame  B']):
    display = np.clip(img ** (1/2.2), 0, 1)   # gamma-correct for display only
    ax.imshow(display)
    ax.set_title(title, fontsize=13)
    ax.axis('off')
plt.tight_layout()
plt.show()

### Step 6 — ECC frame registration

The no-flash and flash photos are taken about 0.5 seconds apart.
The participant may have moved slightly in that time.
We need to align the flash frame pixel-perfectly with the no-flash frame
before multiplying them together.

We use **Enhanced Correlation Coefficient (ECC)** alignment
(Evangelidis & Psarakis, IEEE TPAMI, 2008):

1. Convert both frames to grayscale (luminance)
2. Find the **Euclidean warp** (rotation + translation, no scaling) that
   maximises the normalised correlation between the two grayscale images
3. Apply that warp to the flash frame

After this step: pixel (i, j) in A and B_aligned shows the same physical
point on the face.

In [ ]:
def align_flash_to_noflash(A, B):
    """Warp flash frame B onto no-flash frame A using ECC alignment."""

    # Convert to grayscale using standard luminance weights
    # ECC works on one channel at a time; we align on luminance
    weights = np.array([0.2126, 0.7152, 0.0722])   # Rec. 709 luminance weights
    gray_A  = np.clip(A @ weights, 0, 1).astype(np.float32)
    gray_B  = np.clip(B @ weights, 0, 1).astype(np.float32)

    # Start with the identity warp (assume zero motion)
    warp = np.eye(2, 3, dtype=np.float32)

    # ECC convergence settings
    criteria = (
        cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT,
        500,    # maximum iterations
        1e-6,   # convergence tolerance
    )

    try:
        # Find the warp that maximises cross-correlation
        # MOTION_EUCLIDEAN = rotation + translation only (no shear, no scale)
        cc, warp = cv2.findTransformECC(
            gray_A, gray_B,
            warp,
            cv2.MOTION_EUCLIDEAN,
            criteria,
            None,
            5,    # Gaussian blur radius for robustness
        )
    except cv2.error:
        print('ECC failed — falling back to identity warp (no alignment)')
        cc = 0.0

    # Apply the warp to the full-colour flash frame
    h, w = A.shape[:2]
    B_aligned = cv2.warpAffine(
        B.astype(np.float32), warp, (w, h),
        flags=cv2.INTER_LINEAR + cv2.WARP_INVERSE_MAP,
        borderMode=cv2.BORDER_REPLICATE,
    ).astype(np.float64)

    return B_aligned, warp, float(cc)


print('Aligning flash → no-flash...')
B_aligned, warp_matrix, ecc_score = align_flash_to_noflash(A, B)

print(f'ECC correlation score: {ecc_score:.6f}   (1.0 = perfect)')
print(f'Translation:  Δx = {warp_matrix[0, 2]:.2f} px,  Δy = {warp_matrix[1, 2]:.2f} px')
rotation_deg = np.degrees(np.arctan2(warp_matrix[1, 0], warp_matrix[0, 0]))
print(f'Rotation:     {rotation_deg:.3f}°')

### Step 7 — Skin-mask exposure scaling

The flash frame is globally brighter than the no-flash frame, but we want to
match exposures **over skin pixels only** — because the flash illuminates the
face differently from the rest of the scene.

We:
1. Run **MediaPipe Face Mesh** on the no-flash frame to find the skin region.
   We use the cheek landmarks specifically — the same anatomical region the
   FitSkin scanner measures.
2. Compute the median luminance of both frames over those skin pixels.
3. Scale the aligned flash by $s_{\text{skin}} = \text{median}(A_{\text{skin}}) / \text{median}(B_{\text{skin}})$
   so that B_scaled has the same median luminance as A over skin.

In [ ]:
# MediaPipe Face Mesh cheek landmark indices
# These form a polygon around the left and right cheek regions
CHEEK_LEFT_IDX  = [50, 101, 118, 117, 111, 116, 123, 147, 187, 207,
                   206, 203, 36, 142, 126, 217, 174, 196]
CHEEK_RIGHT_IDX = [280, 352, 347, 346, 340, 345, 352, 376, 411, 427,
                   426, 423, 266, 371, 355, 437, 399, 419]


def make_cheek_mask(img_linear, face_mesh_model):
    """Run MediaPipe and return a boolean cheek mask (True = skin pixel)."""
    h, w = img_linear.shape[:2]

    # MediaPipe expects an sRGB uint8 image
    display = np.clip(img_linear ** (1/2.2), 0, 1)             # gamma for display
    rgb_u8  = (display * 255).astype(np.uint8)                 # float → uint8

    rgb_u8.flags.writeable = False
    result = face_mesh_model.process(rgb_u8)                   # run face mesh

    if not result.multi_face_landmarks:
        raise RuntimeError('No face detected!')

    lm = result.multi_face_landmarks[0].landmark

    def landmark_pts(indices):
        # Convert normalised (0–1) landmark coords to pixel coords
        return np.array([[int(lm[i].x * w), int(lm[i].y * h)]
                          for i in indices], dtype=np.int32)

    # Fill a convex hull around each cheek to get a mask
    mask = np.zeros((h, w), dtype=np.uint8)
    for idx_list in [CHEEK_LEFT_IDX, CHEEK_RIGHT_IDX]:
        pts = landmark_pts(idx_list)
        cv2.fillConvexPoly(mask, cv2.convexHull(pts), 1)

    return mask.astype(bool)


def skin_exposure_scale(A, B_aligned, cheek_mask):
    """Find scalar s so median(s * B_aligned[skin]) == median(A[skin]) in luminance."""
    luma_weights = np.array([0.2126, 0.7152, 0.0722])     # Rec. 709
    luma_A = A[cheek_mask] @ luma_weights                  # luminance of no-flash skin
    luma_B = B_aligned[cheek_mask] @ luma_weights          # luminance of flash skin
    s = float(np.median(luma_A)) / max(float(np.median(luma_B)), 1e-8)
    return s


# Run MediaPipe face mesh
mp_face = mp.solutions.face_mesh
with mp_face.FaceMesh(static_image_mode=True, max_num_faces=1,
                      min_detection_confidence=0.5) as fm:
    cheek_mask = make_cheek_mask(A, fm)

# Compute and apply the skin exposure scale
s_skin     = skin_exposure_scale(A, B_aligned, cheek_mask)
B_scaled   = np.clip(B_aligned * s_skin, 0, None)

print(f'Cheek mask area:  {cheek_mask.sum():,} pixels')
print(f'Skin exposure scale  s_skin = {s_skin:.4f}')
print(f'  (flash was {1/s_skin:.2f}× brighter than no-flash over skin)')

# Visualise the cheek mask overlaid on the no-flash frame
display = np.clip(A ** (1/2.2), 0, 1).copy()
display[cheek_mask] = display[cheek_mask] * 0.5 + np.array([0, 0.4, 0]) * 0.5
plt.figure(figsize=(5, 6))
plt.imshow(display)
plt.title('Cheek mask (green overlay)')
plt.axis('off')
plt.show()

### Step 8 — Geometric-mean reflectance proxy R

This is the core mathematical step. The idea:

- No-flash frame: $A_{\text{pixel}} = \rho \cdot E_{\text{ambient}}$ (skin reflectance × ambient light)
- Flash frame: $B_{\text{pixel}} = \rho \cdot (E_{\text{ambient}} + E_{\text{flash}})$

Taking the geometric mean:
$$\mathbf{R} = \sqrt{\mathbf{A} \odot \mathbf{B}} = \rho \cdot \sqrt{E_{\text{ambient}} \cdot (E_{\text{ambient}} + E_{\text{flash}})}$$

The illumination term does not cancel perfectly, but in practice the flash
dominates over ambient for the skin region, making R a stable proxy for ρ.

We then multiply by the **exposure anchor s_exp** to bring R to a calibrated scale:
$$\mathbf{R}_{\text{anchored}} = s_{\text{exp}} \cdot \mathbf{R}$$

The ⊙ symbol means element-wise multiplication — each pixel's channels
are multiplied independently.

In [ ]:
# Small floor value to avoid sqrt(0) at dark pixels
epsilon = 1e-8

# Geometric-mean reflectance proxy  R = sqrt(A ⊙ B)
R = np.sqrt(np.maximum(A * B_scaled, epsilon))

# Apply the per-participant exposure anchor to bring R to a calibrated scale
R_anchored = np.clip(R * s_exp, 0, None)

print(f'Exposure anchor s_exp = {s_exp:.4f}  (participant {PARTICIPANT})')
print()
print('Mean reflectance over cheek pixels:')
print(f'  R before anchor:  {R[cheek_mask].mean():.4f}')
print(f'  R after  anchor:  {R_anchored[cheek_mask].mean():.4f}')

# Visualise the reflectance map
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, img, title in zip(axes,
                           [A, R_anchored],
                           ['No-flash frame A', 'Reflectance R (anchored)']):
    display = np.clip(img ** (1/2.2), 0, 1)
    ax.imshow(display)
    ax.set_title(title)
    ax.axis('off')
plt.tight_layout()
plt.show()
print('Note: R should look similar to A but with the room lighting cancelled out.')

### Step 9 — Convert R to CIE XYZ then CIELAB

We convert from camera RGB to CIELAB in two steps:

**Step 9a — Camera RGB → XYZ** using the affine matrix M:
$$\begin{bmatrix} X \\ Y \\ Z \end{bmatrix} = M \begin{bmatrix} R \\ G \\ B \\ 1 \end{bmatrix}$$

**Step 9b — XYZ → CIELAB** using the D65 reference white point
$(X_n, Y_n, Z_n) = (0.95047,\; 1.0,\; 1.08883)$:

$$L^* = 116\, f\!\left(\frac{Y}{Y_n}\right) - 16 \qquad
a^* = 500\left[f\!\left(\frac{X}{X_n}\right) - f\!\left(\frac{Y}{Y_n}\right)\right] \qquad
b^* = 200\left[f\!\left(\frac{Y}{Y_n}\right) - f\!\left(\frac{Z}{Z_n}\right)\right]$$

where $f(t) = t^{1/3}$ for $t > (6/29)^3$, else $f(t) = t/(3(6/29)^2) + 4/29$.

CIELAB is perceptually uniform — equal numerical distances correspond to
roughly equal perceived colour differences.

In [ ]:
# D65 reference white (standard illuminant)
D65 = np.array([0.95047, 1.00000, 1.08883])


def camera_rgb_to_lab(rgb_pixels, M, D65=D65):
    """Convert an array of camera-linear RGB pixels to CIELAB."""
    n = rgb_pixels.shape[0]

    # ── Step 9a: RGB → XYZ ────────────────────────────────────────────────
    # Append a column of ones for the affine offset term
    # M is (4, 3): [R, G, B, 1] @ M = [X, Y, Z]
    rgba  = np.hstack([rgb_pixels, np.ones((n, 1))])   # (N, 4)
    xyz   = rgba @ M                                    # (N, 4) @ (4, 3) → (N, 3)
    xyz   = np.maximum(xyz, 0)                          # XYZ must be non-negative

    # ── Step 9b: XYZ → CIELAB ────────────────────────────────────────────
    xyz_norm = xyz / D65    # normalise by reference white

    # Cube-root compression (with linear segment for small values)
    delta = 6.0 / 29.0
    def f(t):
        return np.where(t > delta**3,
                        t ** (1.0 / 3.0),
                        t / (3 * delta**2) + 4.0 / 29.0)

    fx = f(xyz_norm[:, 0])
    fy = f(xyz_norm[:, 1])
    fz = f(xyz_norm[:, 2])

    L = 116 * fy - 16        # lightness  (0 = black, 100 = white)
    a = 500 * (fx - fy)      # red-green axis  (+ = red, − = green)
    b = 200 * (fy - fz)      # yellow-blue axis (+ = yellow, − = blue)

    return np.stack([L, a, b], axis=1)   # (N, 3)


# Extract skin pixels from the anchored reflectance image
skin_rgb = R_anchored[cheek_mask]      # shape: (N_pixels, 3)

# Convert to Lab
skin_lab = camera_rgb_to_lab(skin_rgb, M)

print(f'Converted {len(skin_lab):,} cheek pixels to CIELAB.')
print(f'L* range:  {skin_lab[:,0].min():.1f} – {skin_lab[:,0].max():.1f}')
print(f'a* range:  {skin_lab[:,1].min():.1f} – {skin_lab[:,1].max():.1f}')
print(f'b* range:  {skin_lab[:,2].min():.1f} – {skin_lab[:,2].max():.1f}')

# Show distribution of L* across skin pixels
plt.figure(figsize=(8, 3))
plt.hist(skin_lab[:, 0], bins=40, color='steelblue', edgecolor='white')
plt.xlabel('L* (lightness)')
plt.ylabel('Pixel count')
plt.title('L* distribution over cheek pixels')
plt.tight_layout()
plt.show()

### Step 10 — Cheek ROI: trimmed mean Lab

The skin mask contains some edge pixels that may be contaminated by shadows,
specular highlights, or hair. We compute a **5% trimmed mean** on each axis:
drop the bottom 5% and top 5% of values before averaging.

We also apply a **minimum chroma filter** $C^* = \sqrt{a^{*2} + b^{*2}} \geq 2$
to exclude near-grey pixels that are more likely noise than skin.

The trimmed mean is computed independently for L\*, a\*, and b\*.

In [ ]:
def trimmed_mean_lab(lab, trim_lo=0.05, trim_hi=0.05, min_chroma=2.0):
    """Compute trimmed mean L*, a*, b* over skin pixels."""

    # Remove near-grey pixels: they are likely non-skin noise
    chroma = np.hypot(lab[:, 1], lab[:, 2])   # C* = sqrt(a*^2 + b*^2)
    keep   = chroma >= min_chroma
    lab    = lab[keep]

    if len(lab) == 0:
        raise ValueError('No pixels remaining after chroma filter')

    def trim_axis(arr):
        # Drop bottom trim_lo% and top trim_hi% of values
        lo  = np.percentile(arr, trim_lo * 100)
        hi  = np.percentile(arr, (1 - trim_hi) * 100)
        sel = (arr >= lo) & (arr <= hi)
        return float(arr[sel].mean())

    L = trim_axis(lab[:, 0])
    a = trim_axis(lab[:, 1])
    b = trim_axis(lab[:, 2])
    return L, a, b


L_pred, a_pred, b_pred = trimmed_mean_lab(skin_lab)
C_pred = (a_pred**2 + b_pred**2) ** 0.5   # chroma

print('='*42)
print('PIPELINE PREDICTION  (cheek Lab)')
print('='*42)
print(f'  L*  = {L_pred:6.2f}   (lightness)')
print(f'  a*  = {a_pred:6.2f}   (red–green)')
print(f'  b*  = {b_pred:6.2f}   (yellow–blue)')
print(f'  C*  = {C_pred:6.2f}   (chroma / saturation)')

### Step 11 — Compare to FitSkin scanner: ΔE₀₀

The **FitSkin** contact spectrophotometer measures the cheek directly and
reports Lab values under D65 — the same colour space we computed above.

We compare using **CIEDE2000 (ΔE₀₀)**, the best perceptual colour difference formula:

| ΔE₀₀ | Perception |
|---|---|
| < 1  | Imperceptible |
| 1–3  | Just noticeable |
| 3–6  | Clearly visible (our Phase 4 result: median **3.50**) |
| > 6  | Very obvious |

CIEDE2000 adds corrections for lightness, chroma, and hue non-uniformities
in human vision compared to the older ΔE76 formula.

In [ ]:
def delta_e_2000(lab1, lab2):
    """CIEDE2000 colour difference between two (L*, a*, b*) values."""
    L1, a1, b1 = lab1
    L2, a2, b2 = lab2

    # ── Step 1: adjusted chroma C' and hue angle h' ─────────────────────
    C1   = (a1**2 + b1**2) ** 0.5
    C2   = (a2**2 + b2**2) ** 0.5
    Cavg = (C1 + C2) / 2
    Cavg7 = Cavg**7

    # G factor — corrects a* for chroma non-uniformity
    G   = 0.5 * (1 - (Cavg7 / (Cavg7 + 25**7)) ** 0.5)
    a1p = a1 * (1 + G)
    a2p = a2 * (1 + G)
    C1p = (a1p**2 + b1**2) ** 0.5
    C2p = (a2p**2 + b2**2) ** 0.5

    h1p = np.degrees(np.arctan2(b1, a1p)) % 360
    h2p = np.degrees(np.arctan2(b2, a2p)) % 360

    # ── Step 2: ΔL', ΔC', ΔH' ───────────────────────────────────────────
    dLp = L2 - L1
    dCp = C2p - C1p

    if C1p * C2p == 0:
        dhp = 0
    elif abs(h2p - h1p) <= 180:
        dhp = h2p - h1p
    elif h2p - h1p > 180:
        dhp = h2p - h1p - 360
    else:
        dhp = h2p - h1p + 360

    dHp = 2 * (C1p * C2p) ** 0.5 * np.sin(np.radians(dhp / 2))

    # ── Step 3: weighting functions ──────────────────────────────────────
    Lp_avg = (L1 + L2) / 2
    Cp_avg = (C1p + C2p) / 2
    hp_avg = ((h1p + h2p) / 2
              if abs(h1p - h2p) <= 180
              else (h1p + h2p + 360) / 2)

    T = (1
         - 0.17 * np.cos(np.radians(hp_avg - 30))
         + 0.24 * np.cos(np.radians(2 * hp_avg))
         + 0.32 * np.cos(np.radians(3 * hp_avg + 6))
         - 0.20 * np.cos(np.radians(4 * hp_avg - 63)))

    SL = 1 + 0.015 * (Lp_avg - 50)**2 / (20 + (Lp_avg - 50)**2) ** 0.5
    SC = 1 + 0.045 * Cp_avg
    SH = 1 + 0.015 * Cp_avg * T

    Cp_avg7  = Cp_avg**7
    RC       = 2 * (Cp_avg7 / (Cp_avg7 + 25**7)) ** 0.5
    d_theta  = 30 * np.exp(-((hp_avg - 275) / 25) ** 2)
    RT       = -np.sin(np.radians(2 * d_theta)) * RC

    # ── Final ΔE₀₀ formula ───────────────────────────────────────────────
    de = ((dLp / SL)**2
          + (dCp / SC)**2
          + (dHp / SH)**2
          + RT * (dCp / SC) * (dHp / SH)) ** 0.5
    return float(de)


de00 = delta_e_2000(
    (L_pred, a_pred, b_pred),
    (FITSKIN_L, FITSKIN_A, FITSKIN_B),
)

print('='*42)
print('RESULT')
print('='*42)
print(f'Pipeline:  L*={L_pred:6.2f}  a*={a_pred:6.2f}  b*={b_pred:6.2f}')
print(f'FitSkin:   L*={FITSKIN_L:6.2f}  a*={FITSKIN_A:6.2f}  b*={FITSKIN_B:6.2f}')
print()
print(f'ΔL* = {L_pred - FITSKIN_L:+.2f}   (lightness error)')
print(f'Δa* = {a_pred - FITSKIN_A:+.2f}   (red–green error)')
print(f'Δb* = {b_pred - FITSKIN_B:+.2f}   (yellow–blue error)')
print()
print(f'ΔE₀₀ = {de00:.2f}')

# Interpret the result
if de00 < 1:    interp = 'imperceptible'
elif de00 < 3:  interp = 'just noticeable'
elif de00 < 6:  interp = 'moderate — matches Phase 4 target range (~3.5)'
else:           interp = 'large / clearly visible'
print(f'Interpretation: {interp}')

---
## Summary table

| # | Step | What it does | Key output |
|---|---|---|---|
| 1 | Spectral sensitivity | Monochromator → per-channel camera response curves | q_k(λ) |
| 2 | Flash SPD | Spectroradiometer → flash CCT = 4923 K | flash spectrum |
| 3 | ColorChecker fit | OLS across 24 patches → camera RGB → XYZ | affine M (3×4) |
| 4 | Exposure anchor | White patch → scale to D65 Y=1 | s_exp per participant |
| 5 | DNG demosaic | rawpy, unity WB, linear, norm to 99.5th pct | A, B in [0,1] |
| 6 | ECC registration | Euclidean warp maximising correlation | B_aligned |
| 7 | Skin exposure scale | Median luminance match over cheek pixels | B_scaled |
| 8 | Reflectance proxy | R = √(A ⊙ B) × s_exp | R_anchored |
| 9 | RGB → Lab | M → XYZ, then CIELAB (D65 white) | Lab per pixel |
| 10 | Trimmed mean | 5% trim + chroma filter over cheek ROI | (L*, a*, b*) |
| 11 | CIEDE2000 | Perceptual difference vs FitSkin | **ΔE₀₀** |

**Phase 4 result:** median ΔE₀₀ = **3.50** across N=5 trials (chart-free)

---

## File layout of the repository

| File | Role |
|---|---|
| `flash_no_flash_skin_lab.py` | Main pipeline (production) |
| `run_flash_noflash_skin_lab_raw.sh` | Shell wrapper to run evaluation |
| `train_flash_noflash_checker_calibration.py` | Fit M and s_exp from ColorChecker sessions |
| `build_iphone_calibration_bundle.py` | Assemble the calibration JSON |
| `delta_e_2000.py` | ΔE₀₀ computation |
| `colour_checker_segmentation.py` | OpenCV MCC detector wrapper |
| `exposure_anchor.py` | s_exp derivation |
| `calibration/tier3_affine/` | Pre-fitted calibration files |
| `docs/FLASH_NOFLASH_SKIN_METHODS.md` | Paper methods text |